In [82]:
import pandas as pd

In [83]:
datapath = "data/"

df_history = pd.read_csv(f"{datapath}europe_yearly_full_release_long_format.csv")
df_prices = pd.read_csv(f"{datapath}european_wholesale_electricity_price_data_monthly.csv")

xls = pd.ExcelFile(f"{datapath}targets_download.xlsx")
print(f"found {len(xls.sheet_names)} sheets in targets file: {xls.sheet_names}")

found 4 sheets in targets file: ['raw_data_long', 'sources', 'capacity_target_wide', 'share_target_wide']


In [84]:
df_targets = pd.read_excel(f"{datapath}targets_download.xlsx", sheet_name='raw_data_long')

In [85]:
df_history.head()

,Area,ISO 3 code,Year,Area type,Continent,Ember region,EU,OECD,G20,G7,ASEAN,Category,Subcategory,Variable,Unit,Value,YoY absolute change,YoY % change
0,Austria,AUT,1990,Country or economy,Europe,Europe,1.0,1.0,0.0,0.0,0.0,Electricity demand,Demand,Demand,TWh,48.81,NaN,NaN
1,Austria,AUT,1990,Country or economy,Europe,Europe,1.0,1.0,0.0,0.0,0.0,Electricity demand,Demand per capita,Demand per capita,MWh,6.36,NaN,NaN
2,Austria,AUT,1990,Country or economy,Europe,Europe,1.0,1.0,0.0,0.0,0.0,Electricity generation,Aggregate fuel,Clean,%,66.25,NaN,NaN
3,Austria,AUT,1990,Country or economy,Europe,Europe,1.0,1.0,0.0,0.0,0.0,Electricity generation,Aggregate fuel,Coal,%,12.56,NaN,NaN
4,Austria,AUT,1990,Country or economy,Europe,Europe,1.0,1.0,0.0,0.0,0.0,Electricity generation,Aggregate fuel,Fossil,%,33.75,NaN,NaN


In [86]:
datasets = [df_history, df_targets, df_prices]

for df in datasets:
    print(f"Shape of df: {df.shape}")
    print(f"Available columns in df: {df.columns}")

Shape of df: (69310, 18)
Available columns in df: Index(['Area', 'ISO 3 code', 'Year', 'Area type', 'Continent', 'Ember region',
       'EU', 'OECD', 'G20', 'G7', 'ASEAN', 'Category', 'Subcategory',
       'Variable', 'Unit', 'Value', 'YoY absolute change', 'YoY % change'],
      dtype='str')
Shape of df: (2143, 8)
Available columns in df: Index(['COUNTRY_CODE', 'TARGET_YEAR', 'FUEL_CATEGORY', 'METRIC', 'VALUE',
       'SOURCE_ID', 'NUMBER_OF_UPDATES', 'COLLECTION_YEAR'],
      dtype='str')
Shape of df: (3801, 4)
Available columns in df: Index(['Country', 'ISO3 Code', 'Date', 'Price (EUR/MWhe)'], dtype='str')


In [89]:
df_history_cleaned = df_history[
    (df_history['Year'] == 2023) & 
    (df_history['Category'] == 'Electricity generation') &
    (df_history['Unit'] == 'TWh')
].copy()

df_history_cleaned

,Area,ISO 3 code,Year,Area type,Continent,Ember region,EU,OECD,G20,G7,ASEAN,Category,Subcategory,Variable,Unit,Value,YoY absolute change,YoY % change
1758,Austria,AUT,2023,Country or economy,Europe,Europe,1.0,1.0,0.0,0.0,0.0,Electricity generation,Aggregate fuel,Clean,TWh,60.46,10.04,19.91
1759,Austria,AUT,2023,Country or economy,Europe,Europe,1.0,1.0,0.0,0.0,0.0,Electricity generation,Aggregate fuel,Coal,TWh,0.00,0.00,NaN
1760,Austria,AUT,2023,Country or economy,Europe,Europe,1.0,1.0,0.0,0.0,0.0,Electricity generation,Aggregate fuel,Fossil,TWh,10.82,-3.25,-23.10
1761,Austria,AUT,2023,Country or economy,Europe,Europe,1.0,1.0,0.0,0.0,0.0,Electricity generation,Aggregate fuel,"Hydro, bioenergy and other renewables",TWh,45.23,5.85,14.86
1762,Austria,AUT,2023,Country or economy,Europe,Europe,1.0,1.0,0.0,0.0,0.0,Electricity generation,Aggregate fuel,Renewables,TWh,60.46,10.04,19.91
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
69187,United Kingdom,GBR,2023,Country or economy,Europe,Europe,0.0,1.0,1.0,1.0,0.0,Electricity generation,Fuel,Offshore wind,TWh,49.65,4.54,10.06
69188,United Kingdom,GBR,2023,Country or economy,Europe,Europe,0.0,1.0,1.0,1.0,0.0,Electricity generation,Fuel,Onshore wind,TWh,32.50,-2.60,-7.41
69189,United Kingdom,GBR,2023,Country or economy,Europe,Europe,0.0,1.0,1.0,1.0,0.0,Electricity generation,Fuel,Other fossil,TWh,10.82,-0.25,-2.26
69190,United Kingdom,GBR,2023,Country or economy,Europe,Europe,0.0,1.0,1.0,1.0,0.0,Electricity generation,Fuel,Solar,TWh,13.88,0.54,4.05


In [90]:
df_targets.columns

Index(['COUNTRY_CODE', 'TARGET_YEAR', 'FUEL_CATEGORY', 'METRIC', 'VALUE',
       'SOURCE_ID', 'NUMBER_OF_UPDATES', 'COLLECTION_YEAR'],
      dtype='str')

In [91]:
# identify available target metrics and find the one that matches the history data
print(f"Metrics available in df_targets: {df_targets["METRIC"].unique()}")

# gonna use 'generation_twh'

df_targets_cleaned = df_targets[
    (df_targets["METRIC"] == "generation_twh") &
    (df_targets["TARGET_YEAR"] == 2030)
].copy()

Metrics available in df_targets: <StringArray>
[       'capacity_total_gw', 'share_of_consumption_pct',
           'generation_twh',  'share_of_generation_pct',
   'capacity_additional_mw',               'demand_twh',
               'import_twh']
Length: 7, dtype: str


In [94]:
# pick columns and rename to make the data comparable

df_history_cleaned = df_history_cleaned[["ISO 3 code", "Variable", "Value"]]
df_targets_cleaned = df_targets_cleaned[["COUNTRY_CODE", "FUEL_CATEGORY", "VALUE"]]

df_history_cleaned = df_history_cleaned.rename(columns = {"ISO 3 code": "iso", "Variable": "source", "Value": "value"})
df_targets_cleaned = df_targets_cleaned.rename(columns = {"COUNTRY_CODE": "iso", "FUEL_CATEGORY": "source", "VALUE": "value"})

In [95]:
df_history_cleaned

,iso,source,value
1758,AUT,Clean,60.46
1759,AUT,Coal,0.00
1760,AUT,Fossil,10.82
1761,AUT,"Hydro, bioenergy and other renewables",45.23
1762,AUT,Renewables,60.46
...,...,...,...
69187,GBR,Offshore wind,49.65
69188,GBR,Onshore wind,32.50
69189,GBR,Other fossil,10.82
69190,GBR,Solar,13.88


In [96]:
df_history_cleaned["source"].unique()

<StringArray>
[                                'Clean',
                                  'Coal',
                                'Fossil',
 'Hydro, bioenergy and other renewables',
                            'Renewables',
                                  'Wind',
                        'Wind and solar',
                             'Bioenergy',
                                   'Gas',
                             'Hard coal',
                                 'Hydro',
                               'Lignite',
                               'Nuclear',
                          'Onshore wind',
                          'Other fossil',
                      'Other renewables',
                                 'Solar',
                      'Total generation',
                         'Offshore wind']
Length: 19, dtype: str

In [97]:
df_targets_cleaned["source"].unique()

<StringArray>
[                          'Solar',                            'Wind',
                       'Bioenergy',                           'Hydro',
                'Other Renewables',                            'Coal',
                             'Gas',                         'Nuclear',
                    'Other Fossil',                      'Renewables',
                    'Onshore Wind',                   'Offshore Wind',
                          'Fossil', 'Hydro, bio and other renewables',
                          'Demand',            'Non-hydro renewables',
              'Rest of renewables',                         'Storage',
                    'Pumped hydro',                         'Battery',
                         'Imports']
Length: 21, dtype: str

In [ ]:
# sort the "source" column in fossil and renewables
# note on columns not considered:
# battery/storage/pumped hydro: no info about the source, 

RENEWABLES = {
    "solar", "wind", "onshore wind", "offshore wind", "wind and solar",
    "bioenergy", "hydro", "other renewables", "renewables",
    "hydro, bio and other renewables", "hydro, bioenergy and other renewables",
    "non-hydro renewables", "rest of renewables",
}

FOSSIL = {
    "coal", "hard coal", "lignite", "gas", "other fossil", "fossil",
}

def categorize(value):
    if pd.isna(value):
        return pd.NA
    v = str(value).strip().lower()
    if v in RENEWABLES:
        return "Renewables"
    if v in FOSSIL:
        return "Fossil"
    return "Other"

df_targets_cleaned["category"] = df_targets_cleaned["source"].map(categorize)
df_history_cleaned["category"] = df_history_cleaned["source"].map(categorize)

print(f"Unique values in df_targets_cleaned: {df_targets_cleaned["category"].unique()}")
print(f"Unique values in df_history_cleaned: {df_history_cleaned["category"].unique()}")

Unique values in df_targets_cleaned: <StringArray>
['Renewables', 'Fossil', 'Other']
Length: 3, dtype: str
Unique values in df_history_cleaned: <StringArray>
['Other', 'Fossil', 'Renewables']
Length: 3, dtype: str
